# Demo usage of the common interface for generating embeddings

In [1]:
import sys
import os
import pandas as pd
import numpy as np

# Add project root to sys.path to allow imports
sys.path.append("..")

from helpers.data_loaders import load_movielens_data
from src.models import (
    LightGCNGenerator,
    Node2VecGenerator,
    CleoraGenerator,
    MatrixFactorizationGenerator,
    NCFGenerator
)

/home/patryk/Desktop/put/generic-recommender-system/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Data preprocessing

In [2]:
# Load data
movies_df, ratings_df = load_movielens_data("datasets/movies/movies.csv", "datasets/movies/ratings.csv")

# Preprocess: Filter for high ratings to treat as positive interactions
ratings_df = ratings_df[ratings_df['rating'] >= 4.0].copy()
    
# Create interactions DataFrame
interactions_df = pd.DataFrame({
    'user_id': 'user_' + ratings_df['userId'].astype(str),
    'item_id': 'item_' + ratings_df['movieId'].astype(str),
    'rating': ratings_df['rating'] # Used by Matrix Factorization
})
    
# Use a smaller sample for speed
interactions_df = interactions_df.head(1000)
print(f"Using {len(interactions_df)} interactions for demonstration.")
print(f"Unique users: {interactions_df['user_id'].nunique()}")
print(f"Unique items: {interactions_df['item_id'].nunique()}")

Using 1000 interactions for demonstration.
Unique users: 12
Unique items: 738


## 2. Define Models

In [3]:
models = [
    ("LightGCN", LightGCNGenerator(epochs=2, batch_size=128)),
    ("Node2Vec", Node2VecGenerator(epochs=2, batch_size=128)),
    ("MatrixFactorization", MatrixFactorizationGenerator(epochs=2, batch_size=128)),
    ("NCF", NCFGenerator(epochs=2, batch_size=128)),
    ("Cleora", CleoraGenerator(num_walks=2))
]

## 3. Train and Evaluate

In [4]:
results_dir = 'models'
os.makedirs(results_dir, exist_ok=True)

for name, model in models:
    print(f"\n--- Testing {name} ---")
    try:
        print(f"Training {name}...")
        # Common fit interface
        # Note: Some models use 'rating' column (MF), others ignore it (LightGCN, NCF implicit).
        model.fit(interactions_df, user_col='user_id', item_col='item_id', rating_col='rating')
        
        print(f"Generating embeddings...")
        embeddings = model.get_embeddings()
        
        print(f"Generated {len(embeddings)} embeddings.")
        # Print shape of a sample embedding
        sample_key = list(embeddings.keys())[0]
        print(f"Shape of embedding for '{sample_key}': {embeddings[sample_key].shape}")
        
        # Save model
        save_path = os.path.join(results_dir, f"demo_{name.lower()}_model.pth")
        print(f"Saving model to {save_path}...")
        model.save(save_path)
        
        # Load model (validation)
        print(f"Loading model from {save_path}...")
        model.load(save_path)
        print("Model loaded successfully.")
        
    except ImportError as e:
        print(f"Skipping {name} due to missing dependency: {e}")
    except Exception as e:
        print(f"An error occurred with {name}: {e}")
        import traceback
        traceback.print_exc()



--- Testing LightGCN ---
Training LightGCN...


Epoch 2/2: 100%|██████████| 8/8 [00:00<00:00, 74.06it/s, loss=0.674]
/home/patryk/Desktop/put/generic-recommender-system/notebooks/../src/models/node2vec.py:58: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  edge_index = torch.tensor([edge_index_src, edge_index_dst], dtype=torch.long)


Generating embeddings...
Generated 750 embeddings.
Shape of embedding for 'user_1': (64,)
Saving model to models/demo_lightgcn_model.pth...
Loading model from models/demo_lightgcn_model.pth...
Model loaded successfully.

--- Testing Node2Vec ---
Training Node2Vec...
Generating embeddings...
Generated 750 embeddings.
Shape of embedding for 'item_1': (64,)
Saving model to models/demo_node2vec_model.pth...
Loading model from models/demo_node2vec_model.pth...
Model loaded successfully.

--- Testing MatrixFactorization ---
Training MatrixFactorization...


Epoch 2/2: 100%|██████████| 8/8 [00:00<00:00, 95.35it/s, loss=19]

Generating embeddings...
Generated 750 embeddings.
Shape of embedding for 'user_1': (64,)
Saving model to models/demo_matrixfactorization_model.pth...
Loading model from models/demo_matrixfactorization_model.pth...
Model loaded successfully.

--- Testing NCF ---
Training NCF...



Epoch 2/2: 100%|██████████| 40/40 [00:00<00:00, 246.84it/s, loss=0.377]


Generating embeddings...
Generated 750 embeddings.
Shape of embedding for 'user_1': (64,)
Saving model to models/demo_ncf_model.pth...
Loading model from models/demo_ncf_model.pth...
Model loaded successfully.

--- Testing Cleora ---
Training Cleora...
Generating embeddings...
Generated 750 embeddings.
Shape of embedding for 'item_296': (128,)
Saving model to models/demo_cleora_model.pth...
Loading model from models/demo_cleora_model.pth...
Model loaded successfully.
